# Deux tiers des zéros de zêta simples et sur la ligne critique

Grain **R-12** de la série [Riemann 200](README.md) (EPIC [#20403](https://github.com/jsboige/CoursIA/issues/20403), issue [#23141](https://github.com/jsboige/CoursIA/issues/23141)).

En 2026, Levent Alpöge et Ralph Furman démontrent que **plus des deux tiers des zéros
de la fonction zêta de Riemann sont simples et sur la droite critique**
(*More than two thirds of the zeta zeros are simple and on the critical line*,
arXiv:2608.13637). Le record saute de $0{,}6603$ à $2/3 \approx 0{,}6667$ pour les zéros
simples sur la ligne, et jusqu'à $5/6 \approx 0{,}8333$ pour les zéros distincts —
avec, en fenêtre de Montgomery–Taylor, les constantes optimales $0{,}67250\ldots$
et $0{,}83626\ldots$

**Provenance de l'argument, telle que l'archive la porte** : l'argument a été
*découvert et rédigé par Claude (Anthropic)* ; les auteurs l'ont vérifié ;
la formalisation Lean a été orchestrée par Eric Easley ; le problème avait été
posé par Jarred Sumner. Le projet `anthropics/formal-math` (dossier `zeta23/`)
en livre une formalisation Lean 4 / Mathlib **complète et sans `sorry`**, dont
ce carnet lit le dispositif de confiance sans rien en importer.

Ce carnet mesure plutôt qu'il ne cite : les cinq comptages de zéros et leurs
chaînes d'inégalités, les constantes en jeu recalculées, une mesure à $T$ fini
sur les cent premiers zéros avec hypothèses **pré-enregistrées**, et un **faux
témoin** — le même calcul en ignorant les multiplicités — qui montre le piège
de l'énoncé B. **Ce que ce carnet ne peut pas mesurer** : un énoncé de
$\liminf$ — aucune fenêtre finie ne le confirme ni ne le réfute (hypothèse
H4, pré-enregistrée avant toute mesure).

## Plan

1. **Compter les zéros** : cinq quantités ($N$, $N_d$, $N_0^*$, $N_0^s$, $N^s$) et deux chaînes d'inégalités — mesurées sur un multi-ensemble synthétique.
2. **Les énoncés A à E** : ce que le certificat déclare, en français, avec les noms Lean et les constantes exactes ; la forme ε du $\liminf$.
3. **Le dispositif de confiance** : `Challenge.lean` fiable, `Solution.lean` non fiable, `comparator.json` — ce que « prouvé » veut dire.
4. **La divergence $c=3$ / $c=2$** : le certificat et l'article ne décrivent pas la même voie de preuve ; le carnet la consigne, il ne tranche pas.
5. **Mesure à $T$ fini** : cent premiers zéros, hypothèses pré-enregistrées, verdicts conservés.
6. **Le faux témoin** : ignorer les multiplicités gonfle mécaniquement le rapport.
7. **Le versant Lean** : notre module `LiminfForme.lean`.
8. **Exercices** (trois, à compléter).

In [1]:
# Dependances : stdlib (math) + mpmath.
# mpmath est l exception declaree de la serie (conventions du README :
# R-01, R-02 et R-12 vivent par zetazero, absent de la stdlib).
import math
import mpmath as mp

mp.mp.dps = 30  # 30 decimales : les temoins |zeta(rho)| ~ 1e-30 sont arithmetiques

# Les constantes en jeu, calculees et non recopiees.
DEUX_TIERS = mp.mpf(2) / 3
CINQ_SIXIEMES = mp.mpf(5) / 6
RECORD_ANCIEN_B = mp.mpf('0.6603')   # record precedent pour B (zeros simples sur la ligne)
RECORD_ANCIEN_C = mp.mpf(5) / 12     # record precedent pour C (distincts)

# Fenetre de Montgomery-Taylor (enonce D) : c1* = sqrt(2)*tan(t)/(1+t*tan(t)), t = 1/sqrt(2).
THETA_D = 1 / mp.sqrt(2)
C1_ET = mp.sqrt(2) * mp.tan(THETA_D) / (1 + THETA_D * mp.tan(THETA_D))
MT_LIGNE = 2 - 1 / C1_ET        # constante pour la ligne critique
MT_DISTINCT = (3 - 1 / C1_ET) / 2  # constante pour les zeros distincts

print('c1*              =', mp.nstr(C1_ET, 6), '   (attendu 0.75329...)')
print('2 - 1/c1*        =', mp.nstr(MT_LIGNE, 6), '   (attendu 0.67250...)')
print('(3 - 1/c1*)/2    =', mp.nstr(MT_DISTINCT, 6), '   (attendu 0.83626...)')
print()
print('2/3  =', mp.nstr(DEUX_TIERS, 6), '| 5/6 =', mp.nstr(CINQ_SIXIEMES, 6))
print('records precedents : B > 0.6603, C > 5/12 =', mp.nstr(RECORD_ANCIEN_C, 4))

c1*              = 0.753296    (attendu 0.75329...)
2 - 1/c1*        = 0.672501    (attendu 0.67250...)
(3 - 1/c1*)/2    = 0.83625    (attendu 0.83626...)

2/3  = 0.666667 | 5/6 = 0.833333
records precedents : B > 0.6603, C > 5/12 = 0.4167


## 1. Compter les zéros : cinq quantités, deux chaînes

Le certificat compte les zéros d'ordonnée $T_1 < \gamma \le T_2$ de cinq façons,
dont quatre nous importent ici (notations du README de `zeta23`) :

| Quantité | Sens |
|---|---|
| $N(T_1, T_2)$ | zéros comptés **avec multiplicité** |
| $N_d(T_1, T_2)$ | zéros **distincts** (chaque ordonnée compte une fois) |
| $N_0^*(T_1, T_2)$ | zéros **distincts sur la droite critique** $\operatorname{Re} = 1/2$ |
| $N_0^s(T_1, T_2)$ | zéros **simples et sur la droite critique** |
| $N^s(T_1, T_2)$ | zéros **simples** (où qu'ils soient) |

L'article (§1, chaîne (1.2) de notre copie d'archive) enchaîne :

$$N_0^s \le N_0^* \le N_0 \le N, \qquad N_0^s \le N^s \le N_d \le N.$$

La seconde chaîne dit l'essentiel du piège : **$N_d$ et $N$ ne coïncident pas
dès qu'un zéro est multiple**. Un zéro double compte double dans $N$, une seule
fois dans $N_d$ — et pas du tout dans $N_0^s$. Mesurons cela sur un
multi-ensemble synthétique avant de le rencontrer dans la nature.

In [2]:
# Un multi-ensemble synthetique : 3 simples sur la ligne + 1 zero double sur la ligne.
# multiplicites[i] > 1 = zero multiple a l ordonnee i ; sur_ligne[i] = True.
multiplicites = [1, 2, 1, 1]
sur_ligne     = [True, True, True, True]

def comptages(mult, ligne):
    """Les cinq quantites du certificat sur un multi-ensemble de zéros."""
    N     = sum(mult)                                   # avec multiplicite
    N_d   = len(mult)                                    # distincts
    N0_e  = sum(1 for m, l in zip(mult, ligne) if l)     # distincts sur la ligne
    N0_s  = sum(1 for m, l in zip(mult, ligne) if l and m == 1)  # simples sur la ligne
    N_s   = sum(1 for m in mult if m == 1)               # simples (partout)
    return {'N': N, 'N_d': N_d, 'N0*': N0_e, 'N0s': N0_s, 'Ns': N_s}

c = comptages(multiplicites, sur_ligne)
print('multi-ensemble : ordonnees de multiplicites', multiplicites, ', toutes sur la ligne')
for k, v in c.items():
    print(f'  {k:>4} = {v}')

# Les deux chaines de l article, verifiees sur cet echantillon (elles sont
# elementaires : c est un comptage, pas une analyse).
chaine1 = c['N0s'] <= c['N0*'] <= c['N'] and c['N0*'] <= c['N']   # N0 <= N car N0* <= N
chaine2 = c['N0s'] <= c['Ns'] <= c['N_d'] <= c['N']
assert chaine1 and chaine2, 'les chaines (1.2) doivent tenir sur tout multi-ensemble'
print()
print('chaine N0s <= N0* <= N  :', chaine1)
print('chaine N0s <= Ns <= N_d <= N :', chaine2)
print()
print('ecart N - N_d =', c['N'] - c['N_d'], 'zéros de trop si on confond distincts et comptés :',
      'c est la masse de multiplicite que le faux témoin ignorera en section 6')

multi-ensemble : ordonnees de multiplicites [1, 2, 1, 1] , toutes sur la ligne
     N = 5
   N_d = 4
   N0* = 4
   N0s = 3
    Ns = 3

chaine N0s <= N0* <= N  : True
chaine N0s <= Ns <= N_d <= N : True

ecart N - N_d = 1 zéros de trop si on confond distincts et comptés : c est la masse de multiplicite que le faux témoin ignorera en section 6


## 2. Les énoncés A à E — ce que le certificat déclare

Le README de `zeta23` étiquette ses énoncés ; `Challenge.lean` les porte sous
les noms Lean suivants (chaque énoncé existe aussi en forme *cumulative*) :

| | Énoncé (en français) | Nom Lean (`Challenge.lean`) | Constante |
|---|---|---|---|
| **A** | $\liminf \frac{N_0^*(T, 2T)}{N(T, 2T)} \ge \frac{2}{3}$ — deux tiers des zéros **distincts** sur la ligne | `two_thirds_on_critical_line` (+ `_cumulative`) | $2/3$ |
| **B** | $\liminf \frac{N_0^s(T, 2T)}{N(T, 2T)} \ge \frac{2}{3}$ — deux tiers des zéros **simples et sur la ligne** | `two_thirds_simple_on_critical_line` (+ `_cumulative`) | $2/3$ |
| **C** | $\liminf \frac{N_d(T, 2T)}{N(T, 2T)} \ge \frac{5}{6}$ | `five_sixths_distinct` (+ `_cumulative`) | $5/6$ |
| **D** | fenêtre de Montgomery–Taylor optimale : $\ge 2 - 1/c_1^*$ sur la ligne et $N_d \ge (3 - 1/c_1^*)/2$, avec $c_1^* = \frac{\sqrt{2}\tan\vartheta}{1 + \vartheta\tan\vartheta}$, $\vartheta = 1/\sqrt 2$ | `montgomery_taylor_on_critical_line`, `..._mult` | $0{,}67250\ldots$, $0{,}83626\ldots$ |
| **E** | analogues pour les zéros des $L(s, \chi)$, caractères primitifs modulo $q > 1$ | `dirichlet_*` (cinq déclarations) | — |

**La forme ε du $\liminf$.** Dire $\liminf \frac{X(T)}{N(T)} \ge c$ ne dit rien
de $T$ fixé : cela dit que pour tout $\varepsilon > 0$, il existe $T_0$ tel que

$$\forall T \ge T_0, \quad (c - \varepsilon) \cdot N(T) \le X(T).$$

C'est la forme opérationnelle que le README définit et que notre module Lean
[`Riemann200/LiminfForme.lean`](riemann200_lean/Riemann200/LiminfForme.lean)
démontre par équivalence (`liminf_ge_iff_eps`) puis transpose en comptages
(`proportion_depuis_eps`).

In [3]:
# La forme eps, operationalisee : ce que chaque enonce promet pour eps fixe.
# Et l ordre mesure des constantes (le record precedent est domine par le nouveau).
print('ordre mesure des constantes :')
constantes = [
    ('C ancien (5/12)', mp.mpf(5) / 12),
    ('D  fenetre MT ligne  2 - 1/c1*', MT_LIGNE),
    ('B ancien (0.6603)', RECORD_ANCIEN_B),
    ('A, B nouveaux (2/3)', DEUX_TIERS),
    ('C nouveau (5/6)', CINQ_SIXIEMES),
    ('D  fenetre MT distincts (3-1/c1*)/2', MT_DISTINCT),
]
for nom, v in sorted(constantes, key=lambda p: p[1]):
    print(f'  {mp.nstr(v, 6):>9}   {nom}')

print()
print('ce que B promet, pour eps = 0.01 : il existe T0 tel que pour tout T >= T0,')
print(f'  N0s(T, 2T) >= (2/3 - 0.01) * N(T, 2T) = {mp.nstr(DEUX_TIERS - mp.mpf("0.01"), 4)} * N(T, 2T)')
print('et rien -- strictement rien -- pour un T donne en dessous de T0.')

ordre mesure des constantes :
   0.416667   C ancien (5/12)
     0.6603   B ancien (0.6603)
   0.666667   A, B nouveaux (2/3)
   0.672501   D  fenetre MT ligne  2 - 1/c1*
   0.833333   C nouveau (5/6)
    0.83625   D  fenetre MT distincts (3-1/c1*)/2

ce que B promet, pour eps = 0.01 : il existe T0 tel que pour tout T >= T0,
  N0s(T, 2T) >= (2/3 - 0.01) * N(T, 2T) = 0.6567 * N(T, 2T)
et rien -- strictement rien -- pour un T donne en dessous de T0.


## 3. Le dispositif de confiance — ce que « prouvé » veut dire

Le projet `zeta23` sépare soigneusement trois surfaces, et cette séparation
est la moitié de la leçon :

- **`Challenge.lean` est la surface de confiance** : dix-sept énoncés dont
  `two_thirds_simple_on_critical_line`, énoncés avec `sorry` et n'important
  **que Mathlib**. Lire ce fichier suffit pour savoir *ce qui est affirmé*,
  sans lire la preuve.
- **`Solution.lean` est non fiable** : il délègue à la bibliothèque `Zeta23/`
  (inégalité rang–trace, formules de Weil, comptage de Riemann–von Mangoldt,
  estimates de Stirling, Chebyshev–Mertens, Montgomery–Vaughan). Le README le
  dit : *nothing is assumed* — les théorèmes de tête n'ont pas d'hypothèses,
  le dépôt ne déclare pas d'axiomes, et `#print axioms` ne rapporte que les
  trois axiomes standard de Lean (`propext`, `Classical.choice`, `Quot.sound`).
- **`comparator.json`** dit au registre **Palomar** quelles déclarations des
  deux surfaces doivent coïncider : l'énoncé qu'on lit et la preuve qu'on
  n'a pas lue sont appariés par un tiers.

Pour ce carnet, la conséquence est une règle d'attribution : nous citons les
énoncés **depuis la surface de confiance**, nous ne réimplémentons pas la
preuve, et nous n'importons rien du développement externe (licence Apache-2.0
avec `NOTICE` ; le lac de la série reste nôtre et maître de son build).

## 4. La divergence $c = 3$ / $c = 2$ — consignée, pas tranchée

Le README du certificat déclare **lui-même** que son chemin de preuve pour
l'énoncé C diverge de celui de l'article :

> « Note on Theorem C: in this repository the constant 5/6 is obtained from the
> rank–trace inequality of §3 applied with parameter `c = 3`
> (`Zeta23.ZeroSide.ZeroBlockData.mult_three`); the paper's text derives the
> same 5/6 from Proposition 4.5(iii) with `c = 2`. »

Relue contre **notre copie archivée** de l'article (arXiv:2608.13637v2,
gisement — voir *Références*), cette phrase appelle une précision factuelle :
le numéro **4.5 de la v2 n'est pas une « Proposition »** — c'est un
**Corollaire** à deux claims, sans item (iii) ; l'inégalité rang–trace est le
**Lemme 3.2**, dont les constantes 2 et 4 sont cuites dans l'énoncé (aucun
paramètre `c` libre) ; et c'est la **remarque 7.2** qui porte les faits
voisins : (b) la configuration extrémale $\frac{2}{3}N$ simples +
$\frac{1}{6}N$ doubles atteignant $N_d = \frac{5}{6}N$, (c) les deux niveaux
d'intégralité $(m-1)^2 \ge 0$ et $(m-1)(m-2) \ge 0$, (d) le « Lemme 3.2
à $c = 1$ » (cas $m = 1$ de Chebyshev–Markov–Stieltjes).

Le carnet **consigne** : le certificat et l'exposition ne coïncident pas sur
la voie de preuve de 5/6, et le pointeur « Proposition 4.5(iii) » ne se
résout pas tel quel dans la v2 archivée. Il ne tranche pas : dire laquelle
des deux voies est « la bonne » exigerait une lecture de preuve que ce grain
n'est pas.

## 5. Mesure à $T$ fini — hypothèses pré-enregistrées

Ce que `mpmath.zetazero` sait faire : énumérer les zéros **sur** la ligne
critique, un par un. Ce qu'il ne sait pas : balaier la bande critique (un zéro
hors ligne lui est invisible), ni lire une multiplicité autrement qu'en la
dérivant. Le protocole assumé :

- **témoin de position** : $|\zeta(\rho_k)| \le 10^{-25}$ et
  $\operatorname{Re}(\rho_k) = 1/2$ **tel que l'outil le retourne** ;
- **témoin de simplicité** : $|\zeta'(\rho_k)|$ loin de $0$ — un zéro double
  annulerait aussi $\zeta'$, donc une dérivée $O(1)$ est un témoin *numérique*
  (pas formel) de simplicité ;
- **limite déclarée** : l'énumération rend des ordonnées distinctes ; sans
  témoin de multiplicité, $N$ et $N_d$ sont indistinguables dans la mesure.
  C'est exactement la brèche que le faux témoin de la section 6 exploite.

Les quatre hypothèses sont **fixées avant la mesure**, avec leur critère de
verdict — et les verdicts seront conservés tels quels, y compris REFUTÉ.

In [4]:
# Pre-enregistrement H1-H4 (les predictions sont fixees AVANT les mesures ci-dessous).
N_ZEROS = 100          # les cent premiers zeros non triviaux
TOL_ZERO_H1 = mp.mpf(10) ** -25   # temoin arithmetique : |zeta(rho_k)| <= 1e-25
SEUIL_DERIVEE_H2 = mp.mpf(10) ** -3  # temoin de simplicite : |zeta'(rho_k)| >= 1e-3
SEUIL_PROPORTION_H3 = mp.mpf('0.99')  # proportion N0s/N attendue sur l echantillon

print(f'H1 pre-enregistree : pour k = 1..{N_ZEROS}, |zeta(rho_k)| <= 1e-25 '
      f'et Re(rho_k) = 1/2 (tel que zetazero le retourne)')
print(f"H2 pre-enregistree : pour k = 1..{N_ZEROS}, |zeta'(rho_k)| >= {SEUIL_DERIVEE_H2} "
      f'(temoin numerique de simplicite)')
print(f'H3 pre-enregistree : proportion N0s/N sur les {N_ZEROS} premiers zeros >= {SEUIL_PROPORTION_H3}')
print('H4 pre-enregistree : AUCUNE mesure a T fini ne peut confirmer ni refuter un enonce')
print('     de liminf (les A-E portent sur T -> infty) ; verdict attendu : VRAI par construction.')

H1 pre-enregistree : pour k = 1..100, |zeta(rho_k)| <= 1e-25 et Re(rho_k) = 1/2 (tel que zetazero le retourne)
H2 pre-enregistree : pour k = 1..100, |zeta'(rho_k)| >= 0.001 (temoin numerique de simplicite)
H3 pre-enregistree : proportion N0s/N sur les 100 premiers zeros >= 0.99
H4 pre-enregistree : AUCUNE mesure a T fini ne peut confirmer ni refuter un enonce
     de liminf (les A-E portent sur T -> infty) ; verdict attendu : VRAI par construction.


In [5]:
# Mesure H1 : position. Les cent premiers zeros, temoin |zeta| et Re = 1/2.
zeros = [mp.zetazero(k) for k in range(1, N_ZEROS + 1)]
ecarts_ligne = max(abs(mp.re(rho) - mp.mpf('0.5')) for rho in zeros)
residus = max(abs(mp.zeta(rho)) for rho in zeros)
print(f'k = 1..{N_ZEROS} : gamma_1 = {mp.nstr(zeros[0].imag, 8)}, gamma_{N_ZEROS} = {mp.nstr(zeros[-1].imag, 8)}')
print(f'max |Re(rho_k) - 1/2| = {mp.nstr(ecarts_ligne, 3)}   (tel que zetazero le retourne)')
print(f'max |zeta(rho_k)|      = {mp.nstr(residus, 3)}   (seuil H1 : 1e-25)')
verdict_h1 = 'CONFIRMEE' if (residus <= TOL_ZERO_H1 and ecarts_ligne == 0) else 'REFUTEE'
print(f'Verdict H1 : {verdict_h1}')

k = 1..100 : gamma_1 = 14.134725, gamma_100 = 236.52423
max |Re(rho_k) - 1/2| = 0.0   (tel que zetazero le retourne)
max |zeta(rho_k)|      = 6.05e-29   (seuil H1 : 1e-25)
Verdict H1 : CONFIRMEE


In [6]:
# Mesure H2 : simplicite. Temoins |zeta'(rho_k)| pour k = 1..100.
# Un zero double annulerait zeta ET zeta' ; une derivee O(1) est un temoin
# numerique (pas formel) de simplicite.
derivees = [mp.diff(mp.zeta, rho) for rho in zeros]
min_derivee = min(abs(d) for d in derivees)
print(f"min |zeta'(rho_k)| sur k = 1..{N_ZEROS} = {mp.nstr(min_derivee, 4)}   (seuil H2 : 1e-3)")
print(f"max |zeta'(rho_k)| sur k = 1..{N_ZEROS} = {mp.nstr(max(abs(d) for d in derivees), 4)}")
verdict_h2 = 'CONFIRMEE' if min_derivee >= SEUIL_DERIVEE_H2 else 'REFUTEE'
print(f'Verdict H2 : {verdict_h2}')

min |zeta'(rho_k)| sur k = 1..100 = 0.7932   (seuil H2 : 1e-3)
max |zeta'(rho_k)| sur k = 1..100 = 5.145
Verdict H2 : CONFIRMEE


In [7]:
# Mesure H3 : la proportion N0s/N sur l echantillon, et H4 lue comme conclusion.
# Temoins : les N_ZEROS zeros sont sur la ligne (H1) et simples (H2), donc
# N0s/N = N0*/N = 100/100 sur cet echantillon -- SOUS L'HYPOTHESE declaree que
# les ordonnees enumerees sont distinctes (la mesure ne separe pas N de N_d).
simples_temoins = sum(1 for d in derivees if abs(d) >= SEUIL_DERIVEE_H2)
proportion_mesuree = mp.mpf(simples_temoins) / N_ZEROS
print(f'zeros avec temoin de simplicite : {simples_temoins}/{N_ZEROS}')
print(f'proportion N0s/N mesuree = {mp.nstr(proportion_mesuree, 4)}   (seuil H3 : 0.99)')
verdict_h3 = 'CONFIRMEE' if proportion_mesuree >= SEUIL_PROPORTION_H3 else 'REFUTEE'
print(f'Verdict H3 : {verdict_h3}')
print()
print('H4 : la proportion 1.000 a T ~ ' + mp.nstr(zeros[-1].imag, 3) + ' ne confirme NI ne refute')
print("     l'enonce B (liminf >= 2/3) : un liminf ne dit rien d'un T fixe, il dit que pour")
print("     tout eps il existe T0 au-dela duquel la proportion depasse 2/3 - eps.")
print('     Verdict H4 : VRAI par construction (c est la semantique du liminf,')
print('                  demontree dans LiminfForme.lean::liminf_ge_iff_eps).')

zeros avec temoin de simplicite : 100/100
proportion N0s/N mesuree = 1.0   (seuil H3 : 0.99)
Verdict H3 : CONFIRMEE

H4 : la proportion 1.000 a T ~ 237.0 ne confirme NI ne refute
     l'enonce B (liminf >= 2/3) : un liminf ne dit rien d'un T fixe, il dit que pour
     tout eps il existe T0 au-dela duquel la proportion depasse 2/3 - eps.
     Verdict H4 : VRAI par construction (c est la semantique du liminf,
                  demontree dans LiminfForme.lean::liminf_ge_iff_eps).


**Lecture des verdicts.** H1–H3 : confirmées sur l'échantillon — **dans les
limites déclarées du protocole** : `zetazero` énumère sur la ligne (il ne
prouve pas que les zéros y sont), et le témoin $\zeta'$ est numérique. H4 est
vrai par construction, et c'est le point : la proportion $1{,}000$ observée à
$T \approx 236$ est **compatible avec** un liminf $\ge 2/3$ mais aussi avec
un liminf plus petit — la donnée finie ne contraint pas la queue. C'est
pourquoi les énoncés A–E sont des théorèmes, pas des mesures.

## 6. Le faux témoin — ignorer les multiplicités

L'énoncé B divise par $N(T, 2T)$, compté **avec multiplicité**. Le réflexe
naïf divise par le nombre d'**ordonnées distinctes** — ce que rend une
énumeration comme la nôtre. Sur un échantillon planté d'un zéro double
(99 simples + 1 double = 101 zéros, 100 ordonnées), le faux témoin rapporte
$99/100 = 0{,}99$ là où la vérité est $99/101 \approx 0{,}9802$ : un gonflement
mécanique, d'autant plus grand que la masse de multiplicité est lourde.
C'est le piège structurel de B : **plus il y a de zéros multiples, plus le
faux témoin flatte la proportion**.

In [8]:
# Le faux temoin, execute : 99 simples + 1 zero double, tous sur la ligne.
def rapports(n_simples, n_doubles):
    """Proportion '>= 2/3 ?' calculee correctement (denominateur N, avec
    multiplicite) puis par le faux temoin (denominateur : ordonnees distinctes)."""
    N = n_simples + 2 * n_doubles          # comptage correct, avec multiplicite
    distinctes = n_simples + n_doubles     # ce que rend une enumeration
    correct = mp.mpf(n_simples) / N
    faux = mp.mpf(n_simples) / distinctes
    return correct, faux, distinctes, N

correct, faux, distinctes, N = rapports(99, 1)
print(f'echantillon plante : 99 simples + 1 double -> {N} zeros sur {distinctes} ordonnees')
print(f'  proportion correcte  N0s/N      = 99/{N}  = {mp.nstr(correct, 6)}')
print(f'  faux temoin (ignore les mult.) = 99/{distinctes} = {mp.nstr(faux, 6)}')
print(f'  gonflement = {mp.nstr(faux - correct, 3)}  ({mp.nstr(100 * (faux - correct), 3)} points de pourcentage)')

# L ecart se generalise : s*d / ((s+d)*(s+2d)) pour s simples et d doubles.
s, d = 99, 1
ecart_formule = mp.mpf(s * d) / ((s + d) * (s + 2 * d))
print(f'  formule s*d/((s+d)(s+2d))       = {mp.nstr(ecart_formule, 6)}  (verifie)')

echantillon plante : 99 simples + 1 double -> 101 zeros sur 100 ordonnees
  proportion correcte  N0s/N      = 99/101  = 0.980198
  faux temoin (ignore les mult.) = 99/100 = 0.99
  gonflement = 0.0098  (0.98 points de pourcentage)
  formule s*d/((s+d)(s+2d))       = 0.00980198  (verifie)


## 7. Le versant Lean — notre module `LiminfForme.lean`

Le lac compagnon gagne un module nôtre (issue #23141 : ajout *optionnel et
sans vendoring*) :
`riemann200_lean/Riemann200/LiminfForme.lean`. Il ne prétend pas au théorème A
— Mathlib ne porte ni ζ ni ses zéros (grounding du grain R-01, inchangé). Il
formalise le **langage** de l'énoncé :

| Théorème Lean | Ce qu'il dit |
|---|---|
| `liminf_ge_iff_eps` | $\liminf f \ge c \iff \forall \varepsilon > 0, \exists T_0, \forall T \ge T_0,\ c - \varepsilon < f(T)$ — la forme ε, par équivalence |
| `liminf_ge_mono` | la garantie est héritée par toute constante plus petite : c'est pourquoi $2/3$ domine $0{,}6603$ et $5/12$ |
| `proportion_depuis_eps` | sous $N > 0$ au-delà d'un seuil, la forme ε de la proportion ouvre les fenêtres $(c - \varepsilon) \cdot N(T) \le X(T)$ — la forme exacte du README |

`lake build` vert, `distinct_code_sorry = 0` (l'instrument du dépôt,
`scripts/lean/count_code_sorry.py`). Le développement externe, lui, reste où
il est : lu, cité, jamais importé.

## 8. Exercices

Trois exercices, à compléter dans les cellules suivantes — le carnet s'exécute
de bout en bout même non complété (les cellules d'exercice ne lèvent pas
d'erreur).

In [9]:
# Exercice 1 — a completer : la constante c1* de l enonce D et ses deux fenetres MT.
# Etape 1 : recalculez C1_ET = sqrt(2)*tan(theta)/(1 + theta*tan(theta)) avec theta = 1/sqrt(2)
#           (elle est deja definie plus haut : verifiez votre propre calcul contre elle).
# Etape 2 : en deduire 2 - 1/c1* et (3 - 1/c1*)/2.
# Etape 3 : verifiez que la fenetre MT ligne (0.67250...) est DOMINEE par 2/3 :
#           c est le sens de l enonce D -- optimal en fenetre MT, pas en general.
# Indice : mp.tan, mp.sqrt ; comparez avec MT_LIGNE et MT_DISTINCT definis plus haut.
print('Exercice a completer')
resultat_ex1 = None  # TODO etudiant : l'ecart 2/3 - (2 - 1/c1*)

Exercice a completer


In [10]:
# Exercice 2 — a completer : le faux temoin a taux plantes 1/50, 1/20, 1/10.
# Etape 1 : pour chaque taux t de (1/50, 1/20, 1/10), plantez 100 ordonnees dont
#           round(100*t) portent un zero double (le reste simples), avec rapports().
# Etape 2 : affichez pour chaque taux la proportion correcte, le faux temoin et
#           le gonflement en points de pourcentage.
# Etape 3 : observez la direction : le gonflement croit-il avec le taux ? Vers quel
#           plafond quand tous les ordonnees seraient doubles ?
# Indice : rapports(n_simples, n_doubles) est definie en section 6.
print('Exercice a completer')
resultat_ex2 = None  # TODO etudiant : le gonflement (en pp) pour le taux 1/10

Exercice a completer


In [11]:
# Exercice 3 — a completer : etendre la mesure H2 aux 200 premiers zeros.
# Etape 1 : enumerez les 200 premiers zeros avec mp.zetazero(k).
# Etape 2 : calculez le temoin min |zeta'(rho_k)| sur k = 1..200 (comme en H2).
# Etape 3 : comparez au seuil 1e-3 et donnez le verdict H2bis (CONFIRMEE/REFUTEE).
# Indice : la boucle de la mesure H2 est plus haut ; attendez-vous a ~1 minute de calcul.
print('Exercice a completer')
resultat_ex3 = None  # TODO etudiant : min |zeta'(rho_k)| sur k = 1..200

Exercice a completer


## Limites et bilan honnête

- **`zetazero` calcule *sur* la ligne** : il ne balaie pas la bande critique. Un zéro hors ligne lui est invisible — H1 mesure ce que l'outil retourne, pas une preuve que les zéros y sont. C'est pour cela que A et B restent des **théorèmes**, pas des accumulations de mesures.
- **Le témoin $\zeta'$ est numérique** : $|\zeta'(\rho)| \gg 0$ est une évidence, pas une preuve de simplicité ; un zéro double à dérivée $10^{-9}$ passerait le seuil H2 à 30 décimales près.
- **L'énumération ne sépare pas $N$ de $N_d$** : sans lire les multiplicités, la mesure à $T$ fini vit dans la brèche du faux témoin. La section 6 la quantifie au lieu de la cacher.
- **Aucun $T$ fini ne parle d'un liminf** (H4, pré-enregistrée et vraie par construction) : les proportions mesurées sont compatibles avec l'énoncé B — et avec des liminf bien plus petits.
- **La divergence $c = 3$ / $c = 2$ est consignée, pas tranchée** : le carnet constate que le pointeur « Proposition 4.5(iii) » ne se résout pas dans la v2 archivée (c'est un Corollaire, deux claims, sans (iii)) ; trancher la voie de preuve exigerait une lecture de preuve hors de ce grain.
- **Rien du développement externe n'est importé** : le lac de la série reste nôtre, ses seules dépendances sont Mathlib au pin du lac.

## Références

- L. Alpöge, R. Furman, *More than two thirds of the zeta zeros are simple and on the critical line*, arXiv:2608.13637 (v2). **Copie archivée** : `G:\Mon Drive\MyIA\IA\Bibliographie IA\NumberTheory\2026 - Alpöge & Furman - More than two thirds of the zeta zeros are simple and on the critical line (arXiv 2608.13637).pdf` — toute attribution de ce carnet est lue dans cette copie.
- `anthropics/formal-math`, projet `zeta23` (surface `Challenge.lean`, `Solution.lean`, `comparator.json`, registre Palomar) : https://github.com/anthropics/formal-math/tree/main/zeta23
- Série [Riemann 200](README.md), EPIC [#20403](https://github.com/jsboige/CoursIA/issues/20403) — grains R-01 (le mémoire de 1859) et R-02 (formule explicite) pour le sol calculatoire ; issue [#23141](https://github.com/jsboige/CoursIA/issues/23141) pour ce grain.